In [16]:
import pandas as pd

In [17]:
valisure_ndc_df = pd.read_excel("../HA paper review response/linkdage_data.xlsx", sheet_name="All NDCs From Valisure" ,
    dtype={"NDC": str})

parts = valisure_ndc_df["NDC"].str.split("-", expand=True)

valisure_ndc_df["ndc_11"] = (
    parts[0].str.zfill(5)
    + "-"
    + parts[1].str.zfill(4)
    + "-"
    + parts[2].str.zfill(2)
)


valisure_ndc_df

,NDC,DailyMed Friendly,FEI,Found,ndc_11
0,71093-132-06,71093-132,3008298016,1,71093-0132-06
1,71093-132-04,71093-132,3008298016,1,71093-0132-04
2,71093-132-05,71093-132,3008298016,1,71093-0132-05
3,62037-571-01,62037-571,Not Found,0,62037-0571-01
4,62037-577-10,62037-577,Not Found,0,62037-0577-10
...,...,...,...,...,...
107,68012-004-50,68012-004,3002806613,1,68012-0004-50
108,50228-107-05,50228-107,3008298016,1,50228-0107-05
109,47335-305-88,47335-305,3004561553,1,47335-0305-88
110,49483-621-10,49483-621,3006230648,1,49483-0621-10


In [18]:
sdud_df = pd.read_csv(
    "../processed/2025-12-18-SDUDmonthly.csv",
    dtype={"ndc11": str}
)

In [19]:
# NDC with dash (-) format
sdud_df["ndc_11"] = (
    sdud_df["ndc11"].str[:5]
    + "-"
    + sdud_df["ndc11"].str[5:9]
    + "-"
    + sdud_df["ndc11"].str[9:]
)

In [20]:
sdud_df.columns

Index(['year', 'month', 'quarter', 'ndc11', 'units_reimbursed',
       'num_prescriptions', 'total_amount_reimbursed',
       'medicaid_amount_reimbursed', 'non_medicaid_amount_reimbursed',
       'month_start', 'ndc_11'],
      dtype='object')

In [21]:
sdud_df.head()

,year,month,quarter,ndc11,units_reimbursed,num_prescriptions,total_amount_reimbursed,medicaid_amount_reimbursed,non_medicaid_amount_reimbursed,month_start,ndc_11
0,2019,4,2,00000000275,0.0,0.0,0.0,0.0,0.0,2019-04-01,00000-0002-75
1,2019,5,2,00000000275,0.0,0.0,0.0,0.0,0.0,2019-05-01,00000-0002-75
2,2019,6,2,00000000275,0.0,0.0,0.0,0.0,0.0,2019-06-01,00000-0002-75
3,2019,4,2,00000275107,0.0,0.0,0.0,0.0,0.0,2019-04-01,00000-2751-07
4,2019,5,2,00000275107,0.0,0.0,0.0,0.0,0.0,2019-05-01,00000-2751-07


In [22]:
# Columns to sum (everything except ndc11, year, and non-numeric columns)
sum_cols = [
    "units_reimbursed",
    "num_prescriptions",
    "total_amount_reimbursed",
    "medicaid_amount_reimbursed",
    "non_medicaid_amount_reimbursed",
]

sdud_yearly = (
    sdud_df
    .groupby(["ndc_11", "year"], as_index=False)[sum_cols]
    .sum()
)

sdud_yearly.head()

,ndc_11,year,units_reimbursed,num_prescriptions,total_amount_reimbursed,medicaid_amount_reimbursed,non_medicaid_amount_reimbursed
0,00000-0002-75,2019,0.0,0.0,0.0,0.0,0.0
1,00000-2751-07,2019,0.0,0.0,0.0,0.0,0.0
2,00002-0101-02,2017,0.0,0.0,0.0,0.0,0.0
3,00002-0101-02,2018,0.0,0.0,0.0,0.0,0.0
4,00002-0101-02,2019,0.0,0.0,0.0,0.0,0.0


In [23]:
# Filter sdud and keep the one from valisure
sdud_yearly = sdud_yearly[
    sdud_yearly["ndc_11"].isin(valisure_ndc_df["ndc_11"])
]

# Inner join with valisure_ndc_df
sdud_yearly = sdud_yearly.merge(
    valisure_ndc_df,
    on=["ndc_11"],
    how="inner"
)

# Filter year
sdud_yearly = sdud_yearly[
    sdud_yearly["year"].isin([2020,2022,2024])
]

sdud_yearly

,ndc_11,year,units_reimbursed,num_prescriptions,total_amount_reimbursed,medicaid_amount_reimbursed,non_medicaid_amount_reimbursed,NDC,DailyMed Friendly,FEI,Found
4,00378-6001-91,2020,2.448100e+04,464.0,105570.56,105558.86,11.70,0378-6001-91,0378-6001,Not Found,0
6,00378-6001-91,2022,3.438000e+04,568.0,90906.12,90874.62,31.50,0378-6001-91,0378-6001,Not Found,0
8,00378-6001-91,2024,2.180500e+04,335.0,13220.88,13220.88,0.00,0378-6001-91,0378-6001,Not Found,0
14,00378-7185-05,2020,2.967683e+07,326552.0,2632269.17,2582041.38,50227.79,0378-7185-05,0378-7185,Not Found,0
16,00378-7185-05,2022,4.328166e+07,396785.0,3573830.38,3494759.77,79070.61,0378-7185-05,0378-7185,Not Found,0
...,...,...,...,...,...,...,...,...,...,...,...
727,76385-0128-10,2022,1.200630e+05,1420.0,101173.86,101093.34,80.52,76385-128-10,76385-128,3008763868,1
729,76385-0128-10,2024,3.416510e+05,1770.0,598997.83,598954.83,43.00,76385-128-10,76385-128,3008763868,1
732,76385-0129-01,2020,1.160090e+05,1942.0,15142.30,15019.88,122.42,76385-129-01,76385-129,3008763868,1
734,76385-0129-01,2022,7.752300e+05,13386.0,118689.79,116640.69,2049.10,76385-129-01,76385-129,3008763868,1


In [24]:
# Rename existing sdud_yearly audit columns
sdud_yearly = sdud_yearly.rename(
    columns={
        "units_reimbursed": "audit_units_reimbursed",
        "num_prescriptions": "audit_num_prescriptions",
        "total_amount_reimbursed": "audit_total_amount_reimbursed",
        "medicaid_amount_reimbursed": "audit_medicaid_amount_reimbursed",
        "non_medicaid_amount_reimbursed": "audit_non_medicaid_amount_reimbursed"
    }
)
sdud_yearly

,ndc_11,year,audit_units_reimbursed,audit_num_prescriptions,audit_total_amount_reimbursed,audit_medicaid_amount_reimbursed,audit_non_medicaid_amount_reimbursed,NDC,DailyMed Friendly,FEI,Found
4,00378-6001-91,2020,2.448100e+04,464.0,105570.56,105558.86,11.70,0378-6001-91,0378-6001,Not Found,0
6,00378-6001-91,2022,3.438000e+04,568.0,90906.12,90874.62,31.50,0378-6001-91,0378-6001,Not Found,0
8,00378-6001-91,2024,2.180500e+04,335.0,13220.88,13220.88,0.00,0378-6001-91,0378-6001,Not Found,0
14,00378-7185-05,2020,2.967683e+07,326552.0,2632269.17,2582041.38,50227.79,0378-7185-05,0378-7185,Not Found,0
16,00378-7185-05,2022,4.328166e+07,396785.0,3573830.38,3494759.77,79070.61,0378-7185-05,0378-7185,Not Found,0
...,...,...,...,...,...,...,...,...,...,...,...
727,76385-0128-10,2022,1.200630e+05,1420.0,101173.86,101093.34,80.52,76385-128-10,76385-128,3008763868,1
729,76385-0128-10,2024,3.416510e+05,1770.0,598997.83,598954.83,43.00,76385-128-10,76385-128,3008763868,1
732,76385-0129-01,2020,1.160090e+05,1942.0,15142.30,15019.88,122.42,76385-129-01,76385-129,3008763868,1
734,76385-0129-01,2022,7.752300e+05,13386.0,118689.79,116640.69,2049.10,76385-129-01,76385-129,3008763868,1


In [25]:
# read john's data

john_df = pd.read_excel("metformin data HA Scholar revisions 2026 07 27.xlsx", sheet_name="DATA" ,
    dtype={"NDC": str})

john_df.columns = john_df.columns.str.lower()


parts = john_df["ndc"].str.split("-", expand=True)

john_df["ndc_11"] = (
    parts[0].str.zfill(5)
    + "-"
    + parts[1].str.zfill(4)
    + "-"
    + parts[2].str.zfill(2)
)

john_df.head()

,firm,year,ndc,ndc11,ndc8,strength,ndma (ng/day) valisure,dmf (ng/day) valisure,difference factor,fei_for last inspection,...,sdud_num_prescriptions,sdud_units_reimbursed,total_amount_reimbursed,medicaid_amount_reimbursed,sdud_price_total_per_unit,sdud_price_medicaid_per_unit,nadac_price,countryname,countrycode,ndc_11
0,Alkem Laboratories Limited,2022,67877-413-01,67877041301,67877-413,500,0.0,0.00,NaN,3006370533,...,192479.0,23605394.0,917476.79,900287.72,0.038715,0.037991,0.038213,India,IND,67877-0413-01
1,Alkem Laboratories Limited,2022,67877-413-05,67877041305,67877-413,500,0.0,0.00,NaN,3006370533,...,141153.0,15239426.0,1616631.40,1600388.40,0.106751,0.105704,0.038213,India,IND,67877-0413-05
2,Alkem Laboratories Limited,2022,67877-414-01,67877041401,67877-414,750,42.5,0.00,NaN,3006370533,...,129217.0,9122797.0,1186860.78,1169087.66,0.129934,0.127989,0.076276,India,IND,67877-0414-01
3,Alkem Laboratories Limited,2024,67877-413-01,67877041301,67877-413,500,NaN,151.54,0.261791,3006370533,...,6413.0,845091.0,109767.36,109019.16,0.235369,0.234004,0.029980,India,IND,67877-0413-01
4,Alkem Laboratories Limited,2024,67877-414-01,67877041401,67877-414,750,NaN,151.54,0.000000,3006370533,...,109419.0,8222302.0,1294417.52,1281287.64,0.160179,0.158615,0.064804,India,IND,67877-0414-01


In [26]:
# Filter and keep selected columns only
john_df = john_df[
    [
        "ndc_11",
        "year",
        "sdud_units_reimbursed",
        "sdud_num_prescriptions",
        "total_amount_reimbursed",
        "medicaid_amount_reimbursed",
    ]
]
       

# Rename john_df columns with john_ prefix
john_df = john_df.rename(
    columns={
        col: f"john_{col}" 
        for col in john_df.columns 
        if col not in ["ndc_11", "year"]
    }
)

john_df


,ndc_11,year,john_sdud_units_reimbursed,john_sdud_num_prescriptions,john_total_amount_reimbursed,john_medicaid_amount_reimbursed
0,67877-0413-01,2022,23605394.0,192479.0,917476.79,900287.72
1,67877-0413-05,2022,15239426.0,141153.0,1616631.40,1600388.40
2,67877-0414-01,2022,9122797.0,129217.0,1186860.78,1169087.66
3,67877-0413-01,2024,845091.0,6413.0,109767.36,109019.16
4,67877-0414-01,2024,8222302.0,109419.0,1294417.52,1281287.64
...,...,...,...,...,...,...
113,60687-0640-01,2024,18.0,13.0,15593.07,15593.07
114,68382-0758-01,2024,36882.0,458.0,3719.05,3699.75
115,68382-0759-01,2024,21018.0,293.0,3357.10,3357.10
116,68382-0760-05,2024,38558.0,521.0,9205.72,9040.24


In [27]:
### Audit 1: Duplicates in John data

duplicates = (
    john_df
    .groupby(["ndc_11", "year"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)

duplicates

,ndc_11,year,count


In [28]:
sdud_yearly

,ndc_11,year,audit_units_reimbursed,audit_num_prescriptions,audit_total_amount_reimbursed,audit_medicaid_amount_reimbursed,audit_non_medicaid_amount_reimbursed,NDC,DailyMed Friendly,FEI,Found
4,00378-6001-91,2020,2.448100e+04,464.0,105570.56,105558.86,11.70,0378-6001-91,0378-6001,Not Found,0
6,00378-6001-91,2022,3.438000e+04,568.0,90906.12,90874.62,31.50,0378-6001-91,0378-6001,Not Found,0
8,00378-6001-91,2024,2.180500e+04,335.0,13220.88,13220.88,0.00,0378-6001-91,0378-6001,Not Found,0
14,00378-7185-05,2020,2.967683e+07,326552.0,2632269.17,2582041.38,50227.79,0378-7185-05,0378-7185,Not Found,0
16,00378-7185-05,2022,4.328166e+07,396785.0,3573830.38,3494759.77,79070.61,0378-7185-05,0378-7185,Not Found,0
...,...,...,...,...,...,...,...,...,...,...,...
727,76385-0128-10,2022,1.200630e+05,1420.0,101173.86,101093.34,80.52,76385-128-10,76385-128,3008763868,1
729,76385-0128-10,2024,3.416510e+05,1770.0,598997.83,598954.83,43.00,76385-128-10,76385-128,3008763868,1
732,76385-0129-01,2020,1.160090e+05,1942.0,15142.30,15019.88,122.42,76385-129-01,76385-129,3008763868,1
734,76385-0129-01,2022,7.752300e+05,13386.0,118689.79,116640.69,2049.10,76385-129-01,76385-129,3008763868,1


In [29]:
# Left join: sdud_yearly is the main dataset
sdud_yearly = sdud_yearly.merge(
    john_df,
    on=["ndc_11", "year"],
    how="left"
)


# Create differences
pairs = {
    "units_reimbursed": ("audit_units_reimbursed", "john_sdud_units_reimbursed"),
    "num_prescriptions": ("audit_num_prescriptions", "john_sdud_num_prescriptions"),
    "total_amount_reimbursed": ("audit_total_amount_reimbursed", "john_total_amount_reimbursed"),
    "medicaid_amount_reimbursed": ("audit_medicaid_amount_reimbursed", "john_medicaid_amount_reimbursed"),
}

for name, (audit_col, john_col) in pairs.items():
    sdud_yearly[f"diff_{name}"] = (
        sdud_yearly[audit_col].fillna(0).round(4)
        - sdud_yearly[john_col].fillna(0).round(4)
    )

sdud_yearly


,ndc_11,year,audit_units_reimbursed,audit_num_prescriptions,audit_total_amount_reimbursed,audit_medicaid_amount_reimbursed,audit_non_medicaid_amount_reimbursed,NDC,DailyMed Friendly,FEI,Found,john_sdud_units_reimbursed,john_sdud_num_prescriptions,john_total_amount_reimbursed,john_medicaid_amount_reimbursed,diff_units_reimbursed,diff_num_prescriptions,diff_total_amount_reimbursed,diff_medicaid_amount_reimbursed
0,00378-6001-91,2020,2.448100e+04,464.0,105570.56,105558.86,11.70,0378-6001-91,0378-6001,Not Found,0,NaN,NaN,NaN,NaN,2.448100e+04,464.0,105570.56,105558.86
1,00378-6001-91,2022,3.438000e+04,568.0,90906.12,90874.62,31.50,0378-6001-91,0378-6001,Not Found,0,NaN,NaN,NaN,NaN,3.438000e+04,568.0,90906.12,90874.62
2,00378-6001-91,2024,2.180500e+04,335.0,13220.88,13220.88,0.00,0378-6001-91,0378-6001,Not Found,0,NaN,NaN,NaN,NaN,2.180500e+04,335.0,13220.88,13220.88
3,00378-7185-05,2020,2.967683e+07,326552.0,2632269.17,2582041.38,50227.79,0378-7185-05,0378-7185,Not Found,0,NaN,NaN,NaN,NaN,2.967683e+07,326552.0,2632269.17,2582041.38
4,00378-7185-05,2022,4.328166e+07,396785.0,3573830.38,3494759.77,79070.61,0378-7185-05,0378-7185,Not Found,0,NaN,NaN,NaN,NaN,4.328166e+07,396785.0,3573830.38,3494759.77
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,76385-0128-10,2022,1.200630e+05,1420.0,101173.86,101093.34,80.52,76385-128-10,76385-128,3008763868,1,NaN,NaN,NaN,NaN,1.200630e+05,1420.0,101173.86,101093.34
282,76385-0128-10,2024,3.416510e+05,1770.0,598997.83,598954.83,43.00,76385-128-10,76385-128,3008763868,1,341651.0,1770.0,598997.83,598954.83,0.000000e+00,0.0,0.00,0.00
283,76385-0129-01,2020,1.160090e+05,1942.0,15142.30,15019.88,122.42,76385-129-01,76385-129,3008763868,1,NaN,NaN,NaN,NaN,1.160090e+05,1942.0,15142.30,15019.88
284,76385-0129-01,2022,7.752300e+05,13386.0,118689.79,116640.69,2049.10,76385-129-01,76385-129,3008763868,1,775230.0,13386.0,118689.79,116640.69,0.000000e+00,0.0,0.00,0.00


In [30]:
sdud_yearly.to_csv("audit_results.csv")

In [31]:
valisure_ndc_df = pd.read_excel("../HA paper review response/all_ndc_from_valisure.xlsx", sheet_name="All NDCs From Valisure" ,
    dtype={"NDC": str})

parts = valisure_ndc_df["NDC"].str.split("-", expand=True)

valisure_ndc_df["ndc_11"] = (
    parts[0].str.zfill(5)
    + "-"
    + parts[1].str.zfill(4)
    + "-"
    + parts[2].str.zfill(2)
)


valisure_ndc_df

,NDC,NDC_DailyMed_Friendly,ndc_11
0,0904-7162-61,0904-7162,00904-7162-61
1,0904-7163-61,0904-7163,00904-7163-61
2,0904-7164-61,0904-7164,00904-7164-61
3,70010-063-01,70010-063,70010-0063-01
4,70010-063-05,70010-063,70010-0063-05
...,...,...,...
107,71717-106-11,71717-106,71717-0106-11
108,72578-036-01,72578-036,72578-0036-01
109,75834-500-05,75834-500,75834-0500-05
110,76385-128-10,76385-128,76385-0128-10


In [32]:
valisure_ndc_df.to_csv("clean.csv")